# 차종 기준 재분류 (SUV / 승용차 / 트럭 / 경차) + 새 JSON에 라벨 추가

- 기준: json의 **차종(SmallCategoryId, 예: 쏘나타·어코드·CR-V)** → 아래 매핑표로 새 카테고리 결정
- 저장: 원본 json에 새 필드(`vehicleType`)를 추가한 **새 JSON**을 `DEST_ROOT/<카테고리>/라벨링데이터/...` 에 쓰고, 이미지는 `DEST_ROOT/<카테고리>/원천데이터/...` 로 복사
- 원본(`01.데이터`)은 건드리지 않습니다. json은 항상 **새로 써서** 원본이 지워질 일이 없습니다.

> ⚠️ 매핑표에 **`스파크`가 SUV·경차 중복**입니다. 2번 셀이 중복을 자동 경고하고, `MANUAL_OVERRIDE`로 우선값을 정합니다(기본 경차).
> 실제로 옮기려면 1번 셀에서 **`DRY_RUN = False`**.

## 0. 환경 / 라이브러리

In [1]:
import os, sys, re, json, shutil
from collections import Counter, defaultdict
from pathlib import Path

from tqdm import tqdm

print("Python:", sys.version.split()[0])

Python: 3.11.9


## 1. 경로 & 옵션 + 매핑표 (여기만 수정)

In [2]:
# ==== 경로 3개 ====
IMAGE_ROOT = Path(r"C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\01.데이터\1.Training\원천데이터")
LABEL_ROOT = Path(r"C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\01.데이터\1.Training\라벨링데이터")
DEST_ROOT  = Path(r"C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\차종분류데이터")
# ==================

# --- 옵션 ---
REQUIRE_P00 = True                # True: P00.차량전체 있는 쌍만 / False: 전부
TARGET_CLASS = "P00.차량전체"
NEW_FIELD   = "vehicleType"       # json에 추가할 새 라벨 필드명
SPLIT_BY_CATEGORY = True          # True: 카테고리별 폴더로 분리 / False: 한 곳에 라벨만 추가

DRY_RUN = False                    # True: 미리보기 / False: 실제 실행  ← 실행할 땐 False!
ACTION  = "copy"                  # 이미지 처리: "copy"(원본 보존) / "move"(이동). json은 항상 새로 씀
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

# --- 차종 → 카테고리 매핑표 (이미지 그대로. 필요시 모델 추가/이동) ---
CATEGORY_LISTS = {
    "SUV": ["CR-V","GV80","익스플로러","트레일블레이저","트랙스","트래버스","올란도","볼트EV",
            "X5","X3","GLE-Class","GLC-Class","GLA-Class","Q7","Q5","랭글러","팰리세이드","투싼","코나","아이오닉",
            "싼타페","스타렉스","벨로스터","베뉴","맥스크루즈","넥쏘","i30","클럽맨","Countryman",
            "레인지로버","디스커버리","카니발","쏘울","쏘렌토","스포티지","스토닉","셀토스","모하비","니로",
            "XC60","티구안","골프","티볼리","코란도C","코란도 투리스모","코란도","렉스턴 스포츠","G4 렉스턴",
            "QM6","QM3","XM3","클리오","캡처"],
    "승용차": ["어코드","G90","G80","G70","EQ900","크루즈","임팔라","아베오","말리부","7시리즈","5시리즈",
             "3시리즈","S-Class","E-Class","e-class","CLS-Class","CLA-Class","C-Class","A-Class","A7","A6","A4",
             "엑센트","아반떼","쏘나타","그랜저","알티마","ES","스팅어","K9","K7","K5","K3","S90","파사트",
             "아테온","프리우스","캠리","SM7","SM6","SM5","SM3"],
    "트럭": ["포터2","봉고3"],
    "경차": ["스파크","모닝","레이","다마스"],
}

# --- 중복/모호 차종 우선값 직접 지정 (여기 값이 매핑표보다 우선) ---
MANUAL_OVERRIDE = {
    "스파크": "경차",   # SUV·경차 중복 → 원하는 쪽으로 수정
}

## 2. 매핑 만들기 + 중복 검사

차종 이름이 데이터와 조금 달라도(공백·하이픈 차이) 매칭되도록 정규화 버전도 같이 만듭니다.

In [3]:
# 중복(여러 카테고리에 등장) 검사
appear = defaultdict(set)
for cat, lst in CATEGORY_LISTS.items():
    for m in lst:
        appear[m].add(cat)
conflicts = {m: cats for m, cats in appear.items() if len(cats) > 1}

if conflicts:
    print("⚠️ 중복 차종 (MANUAL_OVERRIDE로 결정됨):")
    for m, cats in conflicts.items():
        chosen = MANUAL_OVERRIDE.get(m, "미지정")
        print(f"  {m}: {sorted(cats)} → 적용: {chosen}")
else:
    print("중복 차종 없음")

# 매핑 확정 (뒤에서 MANUAL_OVERRIDE가 최종 우선)
model_to_cat = {}
for cat, lst in CATEGORY_LISTS.items():
    for m in lst:
        model_to_cat[m] = cat
model_to_cat.update(MANUAL_OVERRIDE)

def norm(s):    # 공백/하이픈/언더바 제거 + 대문자화
    return re.sub(r"[\s\-_]", "", str(s)).upper()

norm_to_cat = {norm(m): c for m, c in model_to_cat.items()}

def category_of(model):
    if model in model_to_cat:
        return model_to_cat[model]
    return norm_to_cat.get(norm(model), "미분류")

print(f"\n등록된 차종 {len(model_to_cat)}개 · 카테고리 {sorted(set(model_to_cat.values()))}")

중복 차종 없음

등록된 차종 101개 · 카테고리 ['SUV', '경차', '승용차', '트럭']


## 2-1. 매핑표 자기 진단: 100종인데 99개만 등록된 이유 찾기

`model_to_cat`은 어디까지나 **매핑표(CATEGORY_LISTS)에 직접 적어 넣은 값**의 개수입니다.
누락 원인은 보통 둘 중 하나입니다.
1. 매핑표를 옮겨 적을 때 한 종이 실수로 빠짐 (매핑표 자체 문제)
2. `스파크`처럼 두 카테고리에 동시에 적혀서 `model_to_cat`(dict)에는 1개로만 집계됨 (딕셔너리 특성상 중복 차종은 1개로 합쳐짐)

아래 셀은 **실제 라벨 json을 전부 스캔**해서 데이터에 진짜 존재하는 차종 목록(정답 100종)을 만든 뒤,
매핑표와 대조해 **① 데이터엔 있는데 매핑표에 없는 차종**과 **② 매핑표엔 있는데 데이터엔 없는 차종**을 각각 보여줍니다.
(라벨 폴더 전체를 훑으므로 몇 분 걸릴 수 있습니다.)

In [4]:
def find_all_files(root: Path, exts: set):
    if not root.exists():
        raise FileNotFoundError(f"경로가 존재하지 않습니다: {root}")
    return [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in exts]

def load_json(path: Path):
    for enc in ("utf-8-sig", "utf-8", "cp949"):
        try:
            return json.loads(path.read_text(encoding=enc))
        except UnicodeDecodeError:
            continue
    raise ValueError(f"인코딩 판별 실패: {path}")

def find_key(obj, key):
    if isinstance(obj, dict):
        if key in obj:
            return obj[key]
        for v in obj.values():
            r = find_key(v, key)
            if r is not None:
                return r
    elif isinstance(obj, list):
        for item in obj:
            r = find_key(item, key)
            if r is not None:
                return r
    return None

_label_files = find_all_files(LABEL_ROOT, {".json"})
print(f"라벨 파일 {len(_label_files):,}개에서 실제 차종(SmallCategoryId) 스캔 중...")

actual_models = Counter()
for p in tqdm(_label_files, desc="차종 스캔"):
    try:
        m = find_key(load_json(p), "SmallCategoryId")
    except Exception:
        continue
    if m:
        actual_models[m.strip()] += 1

actual_set  = set(actual_models)
mapped_set  = set(model_to_cat)          # 최종 매핑(override 반영)에 등록된 차종
listed_set  = set(appear)                # 매핑표(CATEGORY_LISTS)에 적힌 원본 차종(중복 포함 전 집합)

print(f"\n실제 데이터의 차종 수:        {len(actual_set)}종")
print(f"매핑표(CATEGORY_LISTS) 차종 수: {len(listed_set)}종")
print(f"최종 매핑(model_to_cat) 차종 수: {len(mapped_set)}종  (중복 차종은 1개로 집계됨)")

missing_in_map = sorted(actual_set - listed_set)   # ① 데이터엔 있는데 매핑표에 아예 없음 → 진짜 누락
missing_in_data = sorted(listed_set - actual_set)  # ② 매핑표엔 있는데 데이터엔 없음 → 오타 등

print(f"\n① 데이터엔 있는데 매핑표에 없는 차종 ({len(missing_in_map)}개):")
for m in missing_in_map:
    print(f"   {m!r}  (건수: {actual_models[m]:,})")

print(f"\n② 매핑표엔 있는데 데이터엔 없는 차종 ({len(missing_in_data)}개, 오타 의심):")
for m in missing_in_data:
    print(f"   {m!r}")

라벨 파일 257,740개에서 실제 차종(SmallCategoryId) 스캔 중...


차종 스캔: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 257740/257740 [01:11<00:00, 3606.01it/s]


실제 데이터의 차종 수:        101종
매핑표(CATEGORY_LISTS) 차종 수: 101종
최종 매핑(model_to_cat) 차종 수: 101종  (중복 차종은 1개로 집계됨)

① 데이터엔 있는데 매핑표에 없는 차종 (0개):

② 매핑표엔 있는데 데이터엔 없는 차종 (0개, 오타 의심):


## 2-2. "① 누락"이 진짜 누락인지 재확인 + 대표 파일 위치 찾기

①에 뜬 차종이 실제로는 `category_of()`(정규화 비교)로 이미 매핑되고 있을 수 있습니다.
(예: `E-Class` vs `e-class` → 대소문자·하이픈만 다르면 정규화 후 같은 값으로 처리됨)

아래 셀은 ①의 각 차종에 대해
1. `category_of()`로 실제 분류되는 카테고리 (정말 미분류인지 최종 확인)
2. 그 값이 들어있는 **대표 json 파일 경로 1개**
를 출력합니다.

In [5]:
for model in missing_in_map:
    resolved = category_of(model)
    # 해당 값을 가진 라벨 파일 중 하나 찾기
    sample_path = None
    for p in _label_files:
        try:
            v = find_key(load_json(p), "SmallCategoryId")
        except Exception:
            continue
        if v and v.strip() == model:
            sample_path = p
            break

    print(f"차종 {model!r}")
    print(f"  → category_of() 결과: {resolved}  ({'정상 매칭됨 (진단 셀의 오탐)' if resolved != '미분류' else '진짜 미분류 — 매핑표 보완 필요'})")
    print(f"  → 대표 파일: {sample_path}")
    print()

## 3. 파일 수집 + 매칭

In [6]:
def find_all_files(root: Path, exts: set):
    if not root.exists():
        raise FileNotFoundError(f"경로가 존재하지 않습니다: {root}")
    return [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in exts]

image_files = find_all_files(IMAGE_ROOT, IMAGE_EXTS)
label_lookup = {p.stem: p for p in find_all_files(LABEL_ROOT, {".json"})}
paired = [(img, label_lookup[img.stem]) for img in image_files if img.stem in label_lookup]
print(f"이미지 {len(image_files):,} · 매칭된 쌍 {len(paired):,}")

이미지 257,740 · 매칭된 쌍 257,740


## 4. JSON 읽어 P00 여부 + 차종→카테고리 판정

In [7]:
def load_json(path: Path):
    for enc in ("utf-8-sig", "utf-8", "cp949"):
        try:
            return json.loads(path.read_text(encoding=enc))
        except UnicodeDecodeError:
            continue
    raise ValueError(f"인코딩 판별 실패: {path}")

def find_key(obj, key):
    if isinstance(obj, dict):
        if key in obj:
            return obj[key]
        for v in obj.values():
            r = find_key(v, key)
            if r is not None:
                return r
    elif isinstance(obj, list):
        for item in obj:
            r = find_key(item, key)
            if r is not None:
                return r
    return None

def has_target_class(obj):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == "classId" and isinstance(v, str) and v.strip() == TARGET_CLASS:
                return True
            if has_target_class(v):
                return True
    elif isinstance(obj, list):
        return any(has_target_class(i) for i in obj)
    return False

keep = []                    # (img, lbl, category)
unknown_models = Counter()   # 매핑에 없는 차종 (미분류 원인)
n_drop = n_err = 0

for img, lbl in tqdm(paired, desc="판정"):
    try:
        data = load_json(lbl)
    except Exception:
        n_err += 1
        continue
    if REQUIRE_P00 and not has_target_class(data):
        n_drop += 1
        continue
    model = find_key(data, "SmallCategoryId")
    cat = category_of(model)
    if cat == "미분류":
        unknown_models[model if model else "(SmallCategoryId 없음)"] += 1
    keep.append((img, lbl, cat))

print(f"유지 {len(keep):,} · 제외(P00없음) {n_drop:,} · 오류 {n_err:,}")

판정: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 257740/257740 [01:14<00:00, 3442.25it/s]

유지 89,506 · 제외(P00없음) 168,234 · 오류 0


## 5. 카테고리별 요약 + 미분류 차종 목록

In [8]:
by_cat = Counter(c for _, _, c in keep)
order = ["SUV", "승용차", "트럭", "경차", "미분류"]

print("=" * 30)
print(f"{'카테고리':<10}{'쌍 수':>12}")
print("-" * 30)
for c in order:
    if by_cat.get(c):
        print(f"{c:<10}{by_cat[c]:>12,}")
print("-" * 30)
print(f"{'합계':<10}{len(keep):>12,}")
print("=" * 30)

if unknown_models:
    print("\n[미분류] 매핑에 없는 차종 (많은 순):")
    for m, n in unknown_models.most_common():
        print(f"  {m!r}: {n:,}")
    print("→ 1번 셀 CATEGORY_LISTS에 추가하면 분류됩니다.")

카테고리               쌍 수
------------------------------
SUV             40,048
승용차             39,620
트럭               2,096
경차               7,742
------------------------------
합계              89,506


## 6. 실제 실행 (json에 라벨 추가 + 이미지 복사/이동)

`DRY_RUN = True`면 파일을 건드리지 않습니다. 각 json에는 `"{NEW_FIELD}": "<카테고리>"` 가 `rawDataInfo` 안에 추가됩니다.

In [9]:
def dest_of(src: Path, cat: str, kind: str) -> Path:
    # kind: "원천데이터" or "라벨링데이터"
    base = IMAGE_ROOT if kind == "원천데이터" else LABEL_ROOT
    rel = src.relative_to(base)
    return (DEST_ROOT / cat / kind / rel) if SPLIT_BY_CATEGORY else (DEST_ROOT / kind / rel)

print(f"현재 설정 → ACTION={ACTION} · DRY_RUN={DRY_RUN} · SPLIT_BY_CATEGORY={SPLIT_BY_CATEGORY}")

if DRY_RUN:
    print("\n" + "!" * 46)
    print("!! 미리보기 모드 — 아무 파일도 쓰지 않았습니다.")
    print("!! 실행하려면 1번 셀에서 DRY_RUN = False 로 바꾸세요.")
    print("!" * 46)
    for img, lbl, cat in keep[:3]:
        print(f"\n[{cat}] {img.name}")
        print("  이미지 →", dest_of(img, cat, "원천데이터"))
        print("  라벨   →", dest_of(lbl, cat, "라벨링데이터"), f"(+{NEW_FIELD}={cat})")
else:
    ok = skip = fail = 0
    errors = []
    for img, lbl, cat in tqdm(keep, desc="저장"):
        try:
            # 1) 이미지
            img_dst = dest_of(img, cat, "원천데이터")
            if img.exists() and not img_dst.exists():
                img_dst.parent.mkdir(parents=True, exist_ok=True)
                (shutil.copy2 if ACTION == "copy" else shutil.move)(str(img), str(img_dst))
            # 2) 라벨: 새 필드 추가해서 새 json으로 저장
            lbl_dst = dest_of(lbl, cat, "라벨링데이터")
            if not lbl_dst.exists():
                data = load_json(lbl)
                data.setdefault("rawDataInfo", {})[NEW_FIELD] = cat
                lbl_dst.parent.mkdir(parents=True, exist_ok=True)
                lbl_dst.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
            ok += 1
        except Exception as e:
            fail += 1
            errors.append((lbl, str(e)))
    print(f"\n완료 · 처리 {ok:,} / 실패 {fail:,}")
    print("저장 위치:", DEST_ROOT)
    if errors:
        print("실패 예시:", errors[0][0], "→", errors[0][1])

현재 설정 → ACTION=copy · DRY_RUN=False · SPLIT_BY_CATEGORY=True


저장: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 89506/89506 [12:43<00:00, 117.30it/s]


완료 · 처리 89,506 / 실패 0
저장 위치: C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\차종분류데이터


## 7. 결과 확인

In [10]:
if SPLIT_BY_CATEGORY:
    print(f"{'카테고리':<10}{'이미지':>10}{'라벨':>10}")
    print("-" * 30)
    for c in order:
        b = DEST_ROOT / c
        ni = len(find_all_files(b / "원천데이터", IMAGE_EXTS)) if (b / "원천데이터").exists() else 0
        nl = len(find_all_files(b / "라벨링데이터", {".json"})) if (b / "라벨링데이터").exists() else 0
        if ni or nl:
            print(f"{c:<10}{ni:>10,}{nl:>10,}")
else:
    ni = len(find_all_files(DEST_ROOT / "원천데이터", IMAGE_EXTS)) if (DEST_ROOT / "원천데이터").exists() else 0
    nl = len(find_all_files(DEST_ROOT / "라벨링데이터", {".json"})) if (DEST_ROOT / "라벨링데이터").exists() else 0
    print(f"이미지 {ni:,} · 라벨 {nl:,}")

# 샘플 json에 새 필드가 들어갔는지 확인
sample = next((dest_of(l, c, "라벨링데이터") for _, l, c in keep), None)
if sample and sample.exists():
    print(f"\n샘플 {sample.name} 의 {NEW_FIELD} =", find_key(load_json(sample), NEW_FIELD))

카테고리             이미지        라벨
------------------------------
SUV           40,048    40,048
승용차           39,620    39,620
트럭             2,096     2,096
경차             7,742     7,742

샘플 C_211222_AU_006_17_BK_A_T_02_001.json 의 vehicleType = 승용차
